# 01 — Download and quality-check the Allen Human Brain Atlas

The **Allen Human Brain Atlas (AHBA)** is a multimodal, open resource from the Allen Institute for Brain Science that includes genome-wide microarray measurements from spatially localized postmortem tissue samples in six adult human donors. Each tissue sample has anatomical annotations and MNI coordinates.

This makes AHBA appropriate for *From Genetic Risk to Spatial Brain Vulnerability*: it provides the bridge from genes to anatomical location in the healthy adult brain. At this acquisition stage we retain the source microarray data and metadata only. We do **not** reannotate probes, normalize expression, aggregate samples into regions, or perform any disease analysis.

## Reproducible paths and imports

The project root is discovered from `requirements.txt`; no machine-specific path is embedded.

In [1]:
from pathlib import Path
import inspect
import sys

import abagen
import pandas as pd

start = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (start, *start.parents)
    if (candidate / 'requirements.txt').is_file()
)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
import ahba_setup as ahba

AHBA_ROOT = PROJECT_ROOT / 'data' / 'ahba'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURE_DIR = PROJECT_ROOT / 'results' / 'figures'
print(f'Project root: {PROJECT_ROOT}')
print(f'Raw AHBA destination: {AHBA_ROOT}')

Project root: /Users/shibadityadeb/Desktop/brain atlas/brain-disease-vulnerability
Raw AHBA destination: /Users/shibadityadeb/Desktop/brain atlas/brain-disease-vulnerability/data/ahba


## Inspect the installed API before downloading

The installed function signature is inspected at runtime. For `abagen 0.1.3`, the supported acquisition function is `fetch_microarray(data_dir=None, donors=None, resume=True, verbose=1, convert=True, n_proc=1)`. Its documentation specifies that `donors='all'` selects all six AHBA donors.

In [2]:
api_signature = inspect.signature(abagen.fetch_microarray)
print(f'abagen version: {abagen.__version__}')
print(f'abagen.fetch_microarray{api_signature}')
print(inspect.getdoc(abagen.fetch_microarray).split('Returns')[0])

abagen version: 0.1.3
abagen.fetch_microarray(data_dir=None, donors=None, resume=True, verbose=1, convert=True, n_proc=1)
Downloads the Allen Human Brain Atlas microarray expression dataset

Parameters
----------
data_dir : str, optional
    Directory where data should be downloaded and unpacked. Default: $HOME/
    abagen-data
donors : list, optional
    List of donors to download; can be either donor number or UID. Can also
    specify 'all' to download all available donors. Default: 12876
resume : bool, optional
    Whether to resume download of a partly-downloaded file. Default: True
verbose : int, optional
    Verbosity level (0 means no message). Default: 1
convert : bool, optional
    Whether to convert downloaded CSV files into parquet format for faster
    loading in the future; only available if ``fastparquet`` and ``python-
    snappy`` are installed. Default: True
n_proc : int, optional
    Number of processes to parallelize download if multiple donors are
    specified. De

## Download all donors

`resume=True` makes interrupted transfers restartable. `convert=False` preserves the downloaded CSVs without optional parquet conversion. Serial mode (`n_proc=1`) is portable across notebook kernels and avoids macOS process-spawning problems. The returned nested dictionary identifies every donor and important file.

In [3]:
files = abagen.fetch_microarray(
    donors='all',
    data_dir=AHBA_ROOT,
    resume=True,
    verbose=1,
    convert=False,
    n_proc=1,
)
donors = sorted(map(str, files), key=int)
print(f'Available donors ({len(donors)}): {donors}')
for donor in donors:
    print(f'\nDonor {donor}')
    for file_type, path in files[donor].items():
        print(f'  {file_type:11s} {Path(path).name}')

Available donors (6): ['9861', '10021', '12876', '14380', '15496', '15697']

Donor 9861
  microarray  MicroarrayExpression.csv
  ontology    Ontology.csv
  pacall      PACall.csv
  probes      Probes.csv
  annotation  SampleAnnot.csv

Donor 10021
  microarray  MicroarrayExpression.csv
  ontology    Ontology.csv
  pacall      PACall.csv
  probes      Probes.csv
  annotation  SampleAnnot.csv

Donor 12876
  microarray  MicroarrayExpression.csv
  ontology    Ontology.csv
  pacall      PACall.csv
  probes      Probes.csv
  annotation  SampleAnnot.csv

Donor 14380
  microarray  MicroarrayExpression.csv
  ontology    Ontology.csv
  pacall      PACall.csv
  probes      Probes.csv
  annotation  SampleAnnot.csv

Donor 15496
  microarray  MicroarrayExpression.csv
  ontology    Ontology.csv
  pacall      PACall.csv
  probes      Probes.csv
  annotation  SampleAnnot.csv

Donor 15697
  microarray  MicroarrayExpression.csv
  ontology    Ontology.csv
  pacall      PACall.csv
  probes      Probes.csv
 

## File inventory, sizes, and dimensions

The five expected files are `MicroarrayExpression.csv`, `PACall.csv`, `Probes.csv`, `SampleAnnot.csv`, and `Ontology.csv`. The Allen expression and PA-call matrices are headerless; their first column is the probe ID and remaining columns follow sample-annotation row order. Dimensions are counted with a streaming reader so multi-gigabyte matrices do not have to be loaded into memory. The resulting inventory is the requested concise metadata summary.

In [4]:
summary = ahba.inventory(files)
display(summary)
summary_path = PROCESSED_DIR / 'ahba_metadata_summary.csv'
print(f'Saved: {summary_path.relative_to(PROJECT_ROOT)}')

,donor,file_type,filename,relative_path,size_bytes,size_mib,n_rows,n_columns,present
9,10021,annotation,SampleAnnot.csv,data/ahba/microarray/normalized_microarray_don...,84877,0.08,893,13,True
5,10021,microarray,MicroarrayExpression.csv,data/ahba/microarray/normalized_microarray_don...,909763722,867.62,58692,894,True
6,10021,ontology,Ontology.csv,data/ahba/microarray/normalized_microarray_don...,188964,0.18,1839,8,True
7,10021,pacall,PACall.csv,data/ahba/microarray/normalized_microarray_don...,105293448,100.42,58692,894,True
8,10021,probes,Probes.csv,data/ahba/microarray/normalized_microarray_don...,5223610,4.98,58692,7,True
14,12876,annotation,SampleAnnot.csv,data/ahba/microarray/normalized_microarray_don...,36100,0.03,363,13,True
10,12876,microarray,MicroarrayExpression.csv,data/ahba/microarray/normalized_microarray_don...,370091757,352.95,58692,364,True
11,12876,ontology,Ontology.csv,data/ahba/microarray/normalized_microarray_don...,188964,0.18,1839,8,True
12,12876,pacall,PACall.csv,data/ahba/microarray/normalized_microarray_don...,43079928,41.08,58692,364,True
13,12876,probes,Probes.csv,data/ahba/microarray/normalized_microarray_don...,5223610,4.98,58692,7,True


Saved: data/processed/ahba_metadata_summary.csv


## Metadata and acquisition checks

Sample, probe, and ontology tables are loaded. Expression matrices are not transformed. Checks cover the six-donor set, all five file types, nonempty files, MNI coordinates, probe metadata, expression/probe row counts, and expression/sample column counts.

In [5]:
samples, probes, ontology = ahba.load_metadata(files)
issues = ahba.validate(files, summary, samples, probes)
n_genes = ahba.gene_count(probes)

print(f'Combined sample metadata dimensions: {samples.shape}')
print(f'Probe metadata dimensions:          {probes.shape}')
print(f'Ontology dimensions:                {ontology.shape}')
print(f'Unique raw gene symbols:            {n_genes:,}')
print(f'MNI coordinate fields present:      {set(["mni_x", "mni_y", "mni_z"]).issubset(samples.columns)}')
print(f'Expression present:                 {summary.loc[summary.file_type.eq("microarray"), "present"].all()}')
print(f'Probe metadata present:             {summary.loc[summary.file_type.eq("probes"), "present"].all()}')
print(f'Sample metadata present:            {summary.loc[summary.file_type.eq("annotation"), "present"].all()}')
print(f'QC issues:                          {issues or "None"}')
assert not issues, '; '.join(issues)

Combined sample metadata dimensions: (3702, 14)
Probe metadata dimensions:          (58692, 7)
Ontology dimensions:                (1839, 8)
Unique raw gene symbols:            29,131
MNI coordinate fields present:      True
Expression present:                 True
Probe metadata present:             True
Sample metadata present:            True
QC issues:                          None


## AHBA sampling coverage

The first figure places all tissue samples at their supplied MNI coordinates on four anatomical glass-brain projections and colors them by donor. This shows both spatial coverage and the characteristic left-hemisphere sampling emphasis. The second figure reports sample counts by donor.

In [6]:
ahba.make_figures(samples)
coverage_path = FIGURE_DIR / 'ahba_sampling_coverage.png'
counts_path = FIGURE_DIR / 'ahba_samples_by_donor.png'
print(f'Saved {coverage_path.relative_to(PROJECT_ROOT)} ({coverage_path.stat().st_size / 1024:.1f} KiB)')
print(f'Saved {counts_path.relative_to(PROJECT_ROOT)} ({counts_path.stat().st_size / 1024:.1f} KiB)')

Saved results/figures/ahba_sampling_coverage.png (3489.7 KiB)
Saved results/figures/ahba_samples_by_donor.png (89.5 KiB)


## Provenance and final QC report

The provenance record captures source URLs, access date, donor count, data type, preprocessing status, assumptions, and exact direct-package versions. The final report summarizes this acquisition stage and explicitly confirms that regional aggregation has not begun.

In [7]:
provenance_path = ahba.write_provenance(files, n_genes)
print(f'Saved {provenance_path.relative_to(PROJECT_ROOT)}')
ahba.print_qc_report(files, summary, samples, probes, issues)

Saved data/processed/ahba_provenance.md

AHBA ACQUISITION QUALITY-CONTROL REPORT
Number of donors:       6
Total tissue samples:   3,702
Number of probes:       58,692
Genes (raw symbols):    29,131
Major metadata files:   SampleAnnot.csv, Probes.csv, Ontology.csv
Download location:      /Users/shibadityadeb/Desktop/brain atlas/brain-disease-vulnerability/data/ahba/microarray
Missing/suspicious:     None detected
Scope check:            No regional aggregation or disease analysis performed
